In [ ]:
import os

In [ ]:
os.chdir("../")

In [ ]:
import dagshub
dagshub.init(repo_owner='Nilansh-garg', repo_name='fraud-Detection-elliptic-bitcoin', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str

In [ ]:
import os
from src.frauddetection.constants import *
from src.frauddetection.utils.common import read_yaml, create_directories
from pathlib import Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        # Accessing the model_evaluation section from config.yaml
        eval_config = self.config.model_evaluation
        
        create_directories([Path(eval_config.root_dir)])

        evaluation_config = EvaluationConfig(
            path_of_model=Path(eval_config.model_path),
            training_data=Path(eval_config.data_path),
            mlflow_uri="https://dagshub.com/Nilansh-garg/fraud-Detection-elliptic-bitcoin.mlflow", # Keep this or move to a separate secret/config
            all_params=self.params,
            params_in_channels=self.params.IN_CHANNELS,
            params_hidden_channels=self.params.HIDDEN_CHANNELS,
            params_out_channels=self.params.OUT_CHANNELS
        )
        return evaluation_config

In [ ]:
import torch
import mlflow
import mlflow.pytorch
import torch_geometric
from sklearn.metrics import f1_score, precision_score
from src.frauddetection.utils.common import save_json

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _load_model(self, model_class):
        # Using the specific parameters pulled via ConfigurationManager
        model = model_class(
            in_channels=self.config.params_in_channels,
            hidden_channels=self.config.params_hidden_channels,
            out_channels=self.config.params_out_channels
        )
        model.load_state_dict(torch.load(self.config.path_of_model))
        return model

    def evaluation(self, model_class):
        # Registering globals for PyTorch Geometric compatibility
        torch.serialization.add_safe_globals([
            torch_geometric.data.data.DataTensorAttr,
            torch_geometric.data.data.DataEdgeAttr,
            torch_geometric.data.storage.GlobalStorage
        ])
        
        self.model = self._load_model(model_class)
        # Using the path from config.yaml
        self.data = torch.load(self.config.training_data)
        
        self.model.eval()
        with torch.no_grad():
            out = self.model(self.data.x, self.data.edge_index)
            pred = out.argmax(dim=1)
            
            y_true = self.data.y_binary[self.data.test_mask].cpu().numpy()
            y_pred = pred[self.data.test_mask].cpu().numpy()
            
            self.scores = {
                "f1_score": f1_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred)
            }

In [ ]:
import os

# Ensure we are in the root directory (similar to your %pwd and os.chdir logic)
# %pwd 
# os.chdir("../")

try:
    # 1. Initialize Configuration
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    
    # 2. Initialize Evaluation Component
    evaluation = Evaluation(config=eval_config)
    
    # 3. Run Evaluation (Passing the FraudSAGE class architecture)
    evaluation.evaluation(FraudSAGE)
    
    # 4. Save metrics (f1_score, precision) to artifacts/model_evaluation/metrics.json
    evaluation.save_score()
    
    # 5. Log parameters and metrics to MLflow / DagsHub
    evaluation.log_into_mlflow()

except Exception as e:
    raise e